# Phase 1b official run analysis

Notebook per analizzare in profondita `runs/pegst_dvs128gesture_phase1b_official/`.

Obiettivi:
- confrontare le baseline QKFormer Mini a diversi timestep;
- controllare stabilita train/val, best checkpoint e last checkpoint;
- visualizzare accuracy per timestep, confidence ed entropy;
- leggere confusion matrix e per-class accuracy;
- collegare accuracy, firing rate e SOPs stimati;
- analizzare il probe di predicibilita latente confrontando learned predictor, zero, copy previous e linear extrapolation;
- generare insight automatici e tabelle esportabili.

Nota: il notebook e robusto a run mancanti. Se in futuro aggiungi `predictive_aux` o `error_gated` nella stessa cartella, le sezioni finali proveranno a includerle automaticamente.

In [ ]:
from pathlib import Path
import json
import math
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
except Exception:
    sns = None
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except Exception:
        pass

RUN_NAME = "pegst_dvs128gesture_phase1b_official"
RUN_ROOT_CANDIDATES = [
    Path.cwd() / "runs" / RUN_NAME,
    Path.cwd().parent / "runs" / RUN_NAME,
    Path("runs") / RUN_NAME,
    Path("../runs") / RUN_NAME,
]
RUN_ROOT = next((p.resolve() for p in RUN_ROOT_CANDIDATES if p.exists()), None)
if RUN_ROOT is None:
    raise FileNotFoundError(f"Could not find {RUN_NAME}. Tried: {[str(p) for p in RUN_ROOT_CANDIDATES]}")

EXPORT_DIR = Path.cwd() / "phase1b_analysis_exports"
EXPORT_DIR.mkdir(exist_ok=True)

print(f"RUN_ROOT = {RUN_ROOT}")
print(f"EXPORT_DIR = {EXPORT_DIR.resolve()}")

## Data loading helpers

Queste funzioni caricano CSV/JSON, convertono le colonne numeriche e costruiscono un inventario dei run. Sono scritte per non rompersi se una run non contiene, per esempio, `stage_activity.csv` o `prediction_error_best.csv`.

In [ ]:
def read_csv(path: Path) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_csv(path)


def read_json(path: Path):
    if not path.exists() or path.stat().st_size == 0:
        return {}
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def numeric(df: pd.DataFrame, keep=()) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    keep = set(keep)
    for col in out.columns:
        if col not in keep:
            out[col] = pd.to_numeric(out[col], errors="ignore")
    return out


def parse_timestep_from_name(name: str):
    m = re.search(r"_T(\d+)", name)
    return int(m.group(1)) if m else np.nan


def infer_run_kind(name: str) -> str:
    if name.startswith("baseline"):
        return "baseline"
    if "probe" in name:
        return "probe"
    if "error_gated" in name:
        return "error_gated"
    if "predictive" in name or "aux" in name:
        return "predictive_aux"
    return "other"


def has_real_values(df: pd.DataFrame, col: str) -> bool:
    if df.empty or col not in df:
        return False
    s = pd.to_numeric(df[col], errors="coerce")
    return s.notna().any() and float(s.fillna(0).abs().sum()) != 0.0


def load_run(run_dir: Path) -> dict:
    return {
        "path": run_dir,
        "kind": infer_run_kind(run_dir.name),
        "T": parse_timestep_from_name(run_dir.name),
        "metrics": numeric(read_csv(run_dir / "metrics.csv")),
        "metrics_best": read_json(run_dir / "metrics_best.json"),
        "timestep": numeric(read_csv(run_dir / "timestep_metrics.csv")),
        "timestep_best": numeric(read_csv(run_dir / "timestep_metrics_best.csv")),
        "confusion": numeric(read_csv(run_dir / "confusion_matrix.csv")),
        "confusion_best": numeric(read_csv(run_dir / "confusion_matrix_best.csv")),
        "prediction": numeric(read_csv(run_dir / "prediction_error.csv"), keep=("split", "stage", "mode", "method")),
        "prediction_best": numeric(read_csv(run_dir / "prediction_error_best.csv"), keep=("split", "stage", "mode", "method")),
        "prediction_timestep": numeric(read_csv(run_dir / "prediction_timestep_error.csv"), keep=("split", "stage")),
        "prediction_timestep_best": numeric(read_csv(run_dir / "prediction_timestep_error_best.csv"), keep=("split", "stage")),
        "stage_activity": numeric(read_csv(run_dir / "stage_activity.csv"), keep=("network_stage",)),
        "sops": numeric(read_csv(run_dir / "sops_estimate.csv"), keep=("network_stage",)),
        "profile_summary": read_json(run_dir / "profile_summary.json"),
        "params_summary": read_json(run_dir / "params_summary.json"),
        "config_text": (run_dir / "config.yaml").read_text(encoding="utf-8") if (run_dir / "config.yaml").exists() else "",
    }


run_dirs = sorted([p for p in RUN_ROOT.iterdir() if p.is_dir() and not p.name.startswith("_")])
RUNS = {p.name: load_run(p) for p in run_dirs}

inventory = []
for name, data in RUNS.items():
    inventory.append({
        "run": name,
        "kind": data["kind"],
        "T": data["T"],
        "has_metrics": not data["metrics"].empty,
        "has_best": bool(data["metrics_best"]),
        "has_timestep_best": not data["timestep_best"].empty,
        "has_confusion_best": not data["confusion_best"].empty,
        "has_sops": not data["sops"].empty,
        "has_stage_activity": not data["stage_activity"].empty,
        "has_prediction": not data["prediction"].empty,
    })

inventory_df = pd.DataFrame(inventory).sort_values(["kind", "T", "run"], na_position="last")
display(inventory_df)
print(f"Discovered {len(RUNS)} run directories.")

## Baseline summary

Qui confrontiamo i run baseline usando sia il best checkpoint sia il checkpoint finale. Il gap `best_minus_last` e utile per capire quanta instabilita rimane alla fine del training.

In [ ]:
baseline_rows = []
for name, data in RUNS.items():
    m = data["metrics"]
    if data["kind"] != "baseline" or m.empty or "val_acc1" not in m:
        continue
    best_idx = pd.to_numeric(m["val_acc1"], errors="coerce").idxmax()
    best = m.loc[best_idx]
    last = m.iloc[-1]
    best_json = data["metrics_best"] or {}
    baseline_rows.append({
        "run": name,
        "T": int(data["T"]),
        "epochs_completed": len(m),
        "best_epoch": int(best["epoch"]),
        "best_val_acc1": float(best["val_acc1"]),
        "best_train_acc1": float(best.get("acc1", np.nan)),
        "best_val_loss": float(best.get("val_loss", np.nan)),
        "best_lr": float(best.get("lr", np.nan)),
        "last_epoch": int(last["epoch"]),
        "last_val_acc1": float(last["val_acc1"]),
        "last_train_acc1": float(last.get("acc1", np.nan)),
        "last_val_loss": float(last.get("val_loss", np.nan)),
        "best_minus_last": float(best["val_acc1"]) - float(last["val_acc1"]),
        "metrics_best_json_val_acc1": best_json.get("val_acc1", np.nan),
    })

baseline_summary = pd.DataFrame(baseline_rows).sort_values("T")
display(baseline_summary)

if not baseline_summary.empty:
    best_run = baseline_summary.sort_values("best_val_acc1", ascending=False).iloc[0]
    display(Markdown(f"**Best baseline:** `{best_run["run"]}` with best val accuracy **{best_run["best_val_acc1"]:.2f}%** at epoch **{int(best_run["best_epoch"])}**."))

In [ ]:
if not baseline_summary.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    x = np.arange(len(baseline_summary))
    labels = [f"T{int(t)}" for t in baseline_summary["T"]]

    axes[0].bar(x - 0.18, baseline_summary["best_val_acc1"], width=0.36, label="best")
    axes[0].bar(x + 0.18, baseline_summary["last_val_acc1"], width=0.36, label="last")
    axes[0].set_xticks(x, labels)
    axes[0].set_ylim(max(0, baseline_summary[["best_val_acc1", "last_val_acc1"]].min().min() - 5), 100)
    axes[0].set_ylabel("Accuracy (%)")
    axes[0].set_title("Best vs last validation accuracy")
    axes[0].legend()

    axes[1].plot(baseline_summary["T"], baseline_summary["best_val_acc1"], marker="o", label="best val acc")
    axes[1].plot(baseline_summary["T"], baseline_summary["last_val_acc1"], marker="o", label="last val acc")
    axes[1].set_xlabel("Timesteps")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].set_title("Accuracy vs temporal budget")
    axes[1].legend()

    axes[2].bar(labels, baseline_summary["best_minus_last"], color="tab:orange")
    axes[2].axhline(0, color="black", lw=1)
    axes[2].set_ylabel("Best - last accuracy (%)")
    axes[2].set_title("Training stability gap")

    plt.tight_layout()
    plt.show()
else:
    print("No baseline metrics found.")

## Training dynamics

Le curve mostrano train accuracy, validation accuracy, validation loss e learning rate. Nota: con mixup attivo, la train accuracy puo essere meno interpretabile nelle epoche in cui gli input sono miscelati.

In [ ]:
baseline_names = baseline_summary["run"].tolist() if not baseline_summary.empty else []
if baseline_names:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=False)
    for name in baseline_names:
        m = RUNS[name]["metrics"]
        label = f"T{int(RUNS[name]['T'])}"
        axes[0, 0].plot(m["epoch"], m["acc1"], alpha=0.75, label=label)
        axes[0, 1].plot(m["epoch"], m["val_acc1"], alpha=0.9, label=label)
        axes[1, 0].plot(m["epoch"], m["val_loss"], alpha=0.9, label=label)
        if "lr" in m:
            axes[1, 1].plot(m["epoch"], m["lr"], alpha=0.9, label=label)
        best_epoch = int(baseline_summary.loc[baseline_summary["run"] == name, "best_epoch"].iloc[0])
        best_acc = float(baseline_summary.loc[baseline_summary["run"] == name, "best_val_acc1"].iloc[0])
        axes[0, 1].scatter([best_epoch], [best_acc], s=45)

    axes[0, 0].set_title("Train accuracy")
    axes[0, 1].set_title("Validation accuracy with best markers")
    axes[1, 0].set_title("Validation loss")
    axes[1, 1].set_title("Learning rate")
    for ax in axes.ravel():
        ax.set_xlabel("Epoch")
        ax.grid(True, alpha=0.25)
        ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No baseline runs to plot.")

## Timestep behavior

Questa sezione usa `timestep_metrics_best.csv` quando disponibile: mostra come accuracy, confidence ed entropy evolvono mentre il modello accumula timestep.

In [ ]:
timestep_frames = []
for name in baseline_names:
    ts = RUNS[name]["timestep_best"] if not RUNS[name]["timestep_best"].empty else RUNS[name]["timestep"]
    if ts.empty:
        continue
    ts = ts.copy()
    ts["run"] = name
    ts["T"] = int(RUNS[name]["T"])
    ts["relative_timestep"] = ts["timestep"] / ts["T"]
    timestep_frames.append(ts)

timestep_all = pd.concat(timestep_frames, ignore_index=True) if timestep_frames else pd.DataFrame()
display(timestep_all.head())

if not timestep_all.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for name, group in timestep_all.groupby("run"):
        label = f"T{int(group['T'].iloc[0])}"
        axes[0].plot(group["timestep"], group["accuracy"], marker="o", label=label)
        axes[1].plot(group["relative_timestep"], group["accuracy"], marker="o", label=label)
        axes[2].plot(group["timestep"], group["confidence"], marker="o", label=f"{label} conf")
        axes[2].plot(group["timestep"], group["entropy"], marker="x", linestyle="--", label=f"{label} entropy")
    axes[0].set_title("Accuracy by actual timestep")
    axes[1].set_title("Accuracy by relative temporal progress")
    axes[2].set_title("Confidence and entropy")
    axes[0].set_xlabel("Timestep")
    axes[1].set_xlabel("t / T")
    axes[2].set_xlabel("Timestep")
    for ax in axes:
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    early_table = timestep_all.sort_values(["run", "timestep"]).groupby("run").agg(
        T=("T", "first"),
        first_acc=("accuracy", "first"),
        mid_acc=("accuracy", lambda s: s.iloc[len(s)//2]),
        final_acc=("accuracy", "last"),
        final_confidence=("confidence", "last"),
        final_entropy=("entropy", "last"),
    ).reset_index().sort_values("T")
    display(early_table)
else:
    print("No timestep metrics found.")

## Confusion matrix and per-class accuracy

Scegli `SELECTED_RUN` per ispezionare una matrice normalizzata. Per default viene selezionata la baseline con migliore best validation accuracy.

In [ ]:
def confusion_to_matrix(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    mat = df.pivot(index="true_label", columns="pred_label", values="count").fillna(0)
    mat = mat.sort_index().sort_index(axis=1)
    return mat

if not baseline_summary.empty:
    SELECTED_RUN = baseline_summary.sort_values("best_val_acc1", ascending=False).iloc[0]["run"]
else:
    SELECTED_RUN = None

print(f"SELECTED_RUN = {SELECTED_RUN}")

if SELECTED_RUN:
    cm = confusion_to_matrix(RUNS[SELECTED_RUN]["confusion_best"])
    if not cm.empty:
        cm_norm = cm.div(cm.sum(axis=1).replace(0, np.nan), axis=0)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        im = axes[0].imshow(cm, cmap="Blues")
        axes[0].set_title(f"{SELECTED_RUN}: raw confusion")
        axes[0].set_xlabel("Predicted label")
        axes[0].set_ylabel("True label")
        plt.colorbar(im, ax=axes[0], fraction=0.046)

        im = axes[1].imshow(cm_norm, cmap="viridis", vmin=0, vmax=1)
        axes[1].set_title(f"{SELECTED_RUN}: row-normalized confusion")
        axes[1].set_xlabel("Predicted label")
        axes[1].set_ylabel("True label")
        plt.colorbar(im, ax=axes[1], fraction=0.046)
        plt.tight_layout()
        plt.show()

per_class_rows = []
for name in baseline_names:
    cm = confusion_to_matrix(RUNS[name]["confusion_best"])
    if cm.empty:
        continue
    correct = np.diag(cm.values)
    totals = cm.sum(axis=1).values
    for idx, acc in enumerate(correct / np.maximum(totals, 1)):
        per_class_rows.append({"run": name, "T": int(RUNS[name]["T"]), "class": idx, "class_accuracy": acc * 100, "support": totals[idx]})

per_class = pd.DataFrame(per_class_rows)
if not per_class.empty:
    pivot = per_class.pivot_table(index="class", columns="run", values="class_accuracy")
    display(pivot.round(2))
    plt.figure(figsize=(12, 5))
    plt.imshow(pivot, aspect="auto", cmap="magma", vmin=0, vmax=100)
    plt.colorbar(label="Class accuracy (%)")
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=30, ha="right")
    plt.title("Per-class accuracy across baseline best checkpoints")
    plt.tight_layout()
    plt.show()
else:
    print("No confusion matrices found.")

## SOPs, activity and accuracy-cost tradeoff

Questa sezione usa i profili salvati (`sops_estimate.csv` e `stage_activity.csv`) per visualizzare costo stimato, firing rate e densita di attivazione per stage. Alcune run potrebbero non avere profiling, quindi vengono escluse automaticamente.

In [ ]:
sops_frames = []
activity_frames = []
for name in baseline_names:
    sops = RUNS[name]["sops"]
    if not sops.empty:
        s = sops.copy()
        s["run"] = name
        s["T"] = int(RUNS[name]["T"])
        sops_frames.append(s)
    act = RUNS[name]["stage_activity"]
    if not act.empty:
        a = act.copy()
        a["run"] = name
        a["T"] = int(RUNS[name]["T"])
        activity_frames.append(a)

sops_all = pd.concat(sops_frames, ignore_index=True) if sops_frames else pd.DataFrame()
activity_all = pd.concat(activity_frames, ignore_index=True) if activity_frames else pd.DataFrame()

if not sops_all.empty:
    sops_summary = sops_all.groupby(["run", "T"], as_index=False).agg(
        dense_macs=("dense_macs", "sum"),
        estimated_sops=("estimated_sops", "sum"),
        weighted_output_firing_rate=("weighted_output_firing_rate", "mean"),
        output_activity_density=("output_activity_density", "mean"),
    )
    sops_summary = sops_summary.merge(baseline_summary[["run", "best_val_acc1", "last_val_acc1"]], on="run", how="left")
    display(sops_summary.sort_values("T"))

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].plot(sops_summary["estimated_sops"], sops_summary["best_val_acc1"], marker="o", linestyle="")
    for _, r in sops_summary.iterrows():
        axes[0].annotate(f"T{int(r["T"])}", (r["estimated_sops"], r["best_val_acc1"]))
    axes[0].set_xlabel("Estimated SOPs")
    axes[0].set_ylabel("Best val accuracy (%)")
    axes[0].set_title("Accuracy vs estimated SOPs")

    axes[1].bar([f"T{int(t)}" for t in sops_summary["T"]], sops_summary["weighted_output_firing_rate"])
    axes[1].set_title("Mean weighted firing rate")

    axes[2].bar([f"T{int(t)}" for t in sops_summary["T"]], sops_summary["output_activity_density"])
    axes[2].set_title("Mean output activity density")
    for ax in axes:
        ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

    stage_pivot = sops_all.pivot_table(index="network_stage", columns="run", values="estimated_sops", aggfunc="sum").fillna(0)
    display(stage_pivot)
    plt.figure(figsize=(12, 5))
    stage_pivot.plot(kind="bar", stacked=False, figsize=(12, 5))
    plt.ylabel("Estimated SOPs")
    plt.title("Stage-level estimated SOPs")
    plt.tight_layout()
    plt.show()
else:
    print("No SOP profiling files found in these runs.")

## Latent predictability probe

Il probe confronta il predittore allenato offline con tre controlli:
- `zero`: predice tutto zero;
- `copy_previous`: usa `U_t` come stima di `U_{t+1}`;
- `linear_extrapolation`: usa `U_t + alpha * (U_t - U_{t-1})`.

Metriche chiave:
- `normalized_error`: errore normalizzato simmetrico, piu basso e meglio;
- `raw_error_mean`: L1 grezzo medio;
- `prediction_abs_ratio`: rapporto ampiezza predetta / ampiezza target, ideale circa 1.

In [ ]:
probe_path = RUN_ROOT / "probe_from_baseline_T8" / "prediction_error.csv"
probe = numeric(read_csv(probe_path), keep=("stage", "mode", "method"))
print(f"Probe rows: {len(probe)} from {probe_path}")
display(probe.head(12))

if not probe.empty:
    normal_probe = probe[probe["mode"] == "normal"].copy()
    ranking = normal_probe.sort_values(["stage", "history", "normalized_error"])
    display(ranking[["stage", "history", "method", "normalized_error", "raw_error_mean", "prediction_abs_ratio", "loss", "base_loss", "amplitude_loss"]])

    pivot = normal_probe.pivot_table(index=["stage", "history"], columns="method", values="normalized_error")
    for baseline in ["copy_previous", "zero", "linear_extrapolation"]:
        if "learned_predictor" in pivot and baseline in pivot:
            pivot[f"learned_minus_{baseline}"] = pivot["learned_predictor"] - pivot[baseline]
    display(pivot.round(4))
else:
    print("No probe file found.")

In [ ]:
if not probe.empty:
    stages = sorted(probe["stage"].unique())
    modes = sorted(probe["mode"].unique())
    fig, axes = plt.subplots(len(stages), 2, figsize=(16, 5 * len(stages)), squeeze=False)
    for row, stage in enumerate(stages):
        sub = probe[(probe["stage"] == stage) & (probe["mode"].isin(modes))]
        methods = list(dict.fromkeys(sub["method"].tolist()))
        x = np.arange(len(modes))
        width = 0.8 / max(1, len(methods))
        for i, method in enumerate(methods):
            vals = []
            ratios = []
            for mode in modes:
                s = sub[(sub["mode"] == mode) & (sub["method"] == method)]
                vals.append(float(s["normalized_error"].mean()) if not s.empty else np.nan)
                ratios.append(float(s["prediction_abs_ratio"].mean()) if not s.empty else np.nan)
            axes[row, 0].bar(x + (i - len(methods)/2) * width + width/2, vals, width=width, label=method)
            axes[row, 1].bar(x + (i - len(methods)/2) * width + width/2, ratios, width=width, label=method)
        axes[row, 0].set_title(f"{stage}: normalized error by mode/method")
        axes[row, 0].set_xticks(x, modes)
        axes[row, 0].set_ylabel("Normalized error")
        axes[row, 1].set_title(f"{stage}: predicted/target amplitude ratio")
        axes[row, 1].axhline(1.0, color="black", lw=1, linestyle="--")
        axes[row, 1].set_xticks(x, modes)
        axes[row, 1].set_ylabel("Prediction abs ratio")
        for ax in axes[row]:
            ax.grid(True, alpha=0.25)
            ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No probe data to plot.")

## Future predictive/error-gated runs

La cartella attuale contiene baseline e probe. Questa cella cerca comunque eventuali run future con prediction loss, gradient ratio o modulation stats, e produce grafici diagnostici quando le trova.

In [ ]:
diagnostic_runs = []
for name, data in RUNS.items():
    m = data["metrics"]
    if m.empty:
        continue
    pred_cols = [c for c in m.columns if c.startswith("prediction_error_") and c != "prediction_error_mean" and has_real_values(m, c)]
    if pred_cols or has_real_values(m, "grad_ratio_pred_ce") or data["kind"] in {"predictive_aux", "error_gated"}:
        diagnostic_runs.append(name)

print("Diagnostic runs:", diagnostic_runs)

if diagnostic_runs:
    fig, axes = plt.subplots(2, 2, figsize=(16, 9))
    for name in diagnostic_runs:
        m = RUNS[name]["metrics"]
        label = name
        if has_real_values(m, "aux_loss"):
            axes[0, 0].plot(m["epoch"], m["aux_loss"], label=label)
        if has_real_values(m, "prediction_loss_scale"):
            axes[0, 1].plot(m["epoch"], m["prediction_loss_scale"], label=label)
        if has_real_values(m, "grad_ratio_pred_ce"):
            axes[1, 0].plot(m["epoch"], m["grad_ratio_pred_ce"], label=label)
        for col in [c for c in m.columns if c.startswith("prediction_abs_ratio_")]:
            if has_real_values(m, col):
                axes[1, 1].plot(m["epoch"], m[col], label=f"{label}:{col.replace('prediction_abs_ratio_', '')}")
    axes[0, 0].set_title("Weighted aux loss")
    axes[0, 1].set_title("Prediction loss scale")
    axes[1, 0].set_title("Gradient ratio pred/CE")
    axes[1, 0].axhspan(0.05, 0.20, alpha=0.15, color="green", label="target 0.05-0.20")
    axes[1, 0].axhline(0.30, color="red", linestyle="--", lw=1, label="risk > 0.30")
    axes[1, 1].set_title("Prediction amplitude ratio")
    axes[1, 1].axhline(1.0, color="black", linestyle="--", lw=1)
    for ax in axes.ravel():
        ax.set_xlabel("Epoch")
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No predictive_aux/error_gated diagnostic metrics found in this run folder yet.")

## Automatic insight report

Questa cella genera un breve report testuale. Non sostituisce l'analisi visiva, ma aiuta a non perdere i segnali principali.

In [ ]:
insights = []

if not baseline_summary.empty:
    best = baseline_summary.sort_values("best_val_acc1", ascending=False).iloc[0]
    most_stable = baseline_summary.sort_values("best_minus_last", ascending=True).iloc[0]
    insights.append(f"Best baseline: {best["run"]} with {best["best_val_acc1"]:.2f}% best val accuracy at epoch {int(best["best_epoch"])}.")
    insights.append(f"Smallest best-last gap: {most_stable["run"]} with {most_stable["best_minus_last"]:.2f} percentage points.")
    if best["best_epoch"] < 75:
        insights.append("The best checkpoint occurs before/around mixup-off for at least the top run; train accuracy at that epoch can be hard to interpret because mixed inputs are evaluated against hard labels.")
    if baseline_summary["best_minus_last"].max() > 3:
        row = baseline_summary.sort_values("best_minus_last", ascending=False).iloc[0]
        insights.append(f"Largest stability gap: {row["run"]} loses {row["best_minus_last"]:.2f} points from best to last checkpoint. Best-checkpoint evaluation is important.")

if not timestep_all.empty:
    final_by_t = timestep_all.sort_values("T").groupby("run").tail(1)
    fastest = final_by_t.sort_values("accuracy", ascending=False).iloc[0]
    insights.append(f"Best final timestep accuracy among timestep-best files: {fastest["run"]} reaches {fastest["accuracy"]:.2f}% at timestep {int(fastest["timestep"])}.")

if not probe.empty:
    normal = probe[probe["mode"] == "normal"]
    for stage, sub in normal.groupby("stage"):
        best_method = sub.sort_values("normalized_error").iloc[0]
        insights.append(f"Probe {stage}/normal: best normalized error is {best_method["normalized_error"]:.3f} from {best_method["method"]}; amplitude ratio {best_method["prediction_abs_ratio"]:.2f}.")
        learned = sub[sub["method"] == "learned_predictor"]
        copy = sub[sub["method"] == "copy_previous"]
        if not learned.empty and not copy.empty:
            delta = float(learned["normalized_error"].iloc[0]) - float(copy["normalized_error"].iloc[0])
            sign = "beats" if delta < 0 else "does not beat"
            insights.append(f"Probe {stage}/normal: learned predictor {sign} copy_previous by {abs(delta):.3f} normalized-error points.")
        amp = float(best_method["prediction_abs_ratio"])
        if amp < 0.7 or amp > 1.3:
            insights.append(f"Probe {stage}/normal: amplitude ratio is far from 1.0; watch for scale mismatch/collapse or over-amplification.")

if insights:
    display(Markdown("\n".join(f"- {x}" for x in insights)))
else:
    print("No insights generated; check that data files were loaded correctly.")

## Export compact tables

Questa cella salva alcune tabelle derivate nella directory `phase1b_analysis_exports/`, utile se vuoi allegarle a report o confrontarle tra run.

In [ ]:
exports = {
    "baseline_summary.csv": baseline_summary,
    "timestep_all.csv": timestep_all,
    "sops_summary.csv": sops_summary if "sops_summary" in globals() else pd.DataFrame(),
    "probe_prediction_error.csv": probe,
    "probe_normal_pivot.csv": pivot.reset_index() if "pivot" in globals() and isinstance(pivot, pd.DataFrame) else pd.DataFrame(),
    "per_class_accuracy.csv": per_class if "per_class" in globals() else pd.DataFrame(),
}
for filename, df in exports.items():
    if isinstance(df, pd.DataFrame) and not df.empty:
        path = EXPORT_DIR / filename
        df.to_csv(path, index=False)
        print(f"wrote {path}")